# Inputs

## Common inputs

In [ ]:
# Set your version number here (see instructions above)

version = 9

In [ ]:
import pandas as pd
import os
import ast
from pathlib import Path
import fleet_adoption_game as game_model
import load_data as load
import importlib
from default_assumptions import FUEL_GROUPS_AVAIL
import concurrent.futures

## Functions

In [ ]:
def convert_columns_to_dict(user_params):
    # Convert string representations of Python literals (e.g., sets/lists) into objects
    columns_to_parse = ['retrofit_options', 'newbuild_options', 'allowed_fuels', 'reward_fuels']

    for column in columns_to_parse:
        if column not in user_params:
            continue

        value = user_params[column]

        # Normalize missing values to empty list
        if pd.isna(value):
            user_params[column] = []
            continue

        if isinstance(value, str):
            value = value.strip()

            # Normalize blank/none-like strings to empty list
            if not value or value.lower() in {'nan', 'none'}:
                user_params[column] = []
                continue

            try:
                user_params[column] = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                pass

    return user_params

In [ ]:
# Load the scenario data from the Excel file

def load_scenario_data(file_path, scenarios_to_run_tab):
    # Load the 'scenarios' sheet
    scenario_data = pd.read_excel(file_path, sheet_name=scenarios_to_run_tab)

    # Remove leading and trailing whitespace from column names
    scenario_data.columns = scenario_data.columns.str.strip()

    # Remove leading and trailing whitespace from all string values in the dataframe
    scenario_data = scenario_data.map(lambda x: x.strip() if isinstance(x, str) else x)
    return scenario_data

# Simulate the model run for each scenario
def run_scenario(scenario_params, RUN_OUTPUT_DIR, model_excel_db, round_nb):
    print(f"Running scenario with parameters: {scenario_params}")

    # Extract all the necessary parameters from user_params
    user_params = scenario_params  # The scenario parameters come from the Excel sheet

    # Convert list columns in user_params to proper sets
    user_params = convert_columns_to_dict(user_params)

    # Extract other parameters from the Excel sheet
    start_year = user_params['start_year']
    end_year = user_params['end_year']
    planning_horizon_years = user_params['planning_horizon_years']
    ENGINE_PEER_FACTOR = user_params['ENGINE_PEER_FACTOR']
    SHAPE_PEER_INFLUENCE = user_params['SHAPE_PEER_INFLUENCE']
    critical_mass_engine = user_params['critical_mass_engine']
    slope_engine = user_params['slope_engine']
    max_iterations = user_params['max_iterations']
    su_market_flag = user_params['su_market_flag']
    num_players = user_params['num_players']

    # Create scenario_name-specific output directory
    scenario_number = scenario_params['scenario_number']
    scenario_output_dir =  Path(RUN_OUTPUT_DIR) / "zip_files" / f"scenario_{scenario_number}"
    scenario_output_dir.mkdir(parents=True, exist_ok=True)

    # --- INPUT DATA ---
    engine_fuels_dict, df_drop_in_fuels_ref = load.load_engine_fuels_dict(model_excel_db, user_params['allowed_fuels'])

    ref = load.load_reference_data(model_excel_db)
    df_imo_targets_data = ref["df_imo_targets_data"]
    df_fueleu_targets_data = ref["df_fueleu_targets_data"]
    ru_price_data = ref["ru_price_data"]
    initial_su_price_data = ref["initial_su_price_data"]
    fuel_price_data = ref["fuel_price_data"]
    df_wtt_carbon_intensity_ref = ref["df_wtt_carbon_intensity_ref"]
    df_reward_ref = ref["df_reward_ref"]
    df_tc_rates_ref = ref["df_tc_rates_ref"]
    df_operational_data_ref = ref["df_operational_data_ref"]
    df_energy_density_ref = ref["df_energy_density_ref"]
    df_pilot_fuel_input_ref = ref["df_pilot_fuel_input_ref"]
    engines_design_mapping = ref["engines_design_mapping"]
    fuel_engine_mapping = ref["fuel_engine_mapping"]
    df_ttw_carbon_intensity_ref = ref["df_ttw_carbon_intensity_ref"]
    imo_size_map = ref["imo_size_map"]
    df_scrap_rates_ref = ref["df_scrap_rates_ref"]
    (df_opex_conventional_ref, df_opex_nonconv_wages_ref, df_opex_nonconv_insurance_ref,
     df_opex_nonconv_ssf_ref, df_opex_nonconv_isf_ref, df_opex_nonconv_spares_ref) = ref["df_opex_data"]
    df_growth_infra = ref["df_growth_infra"]
    df_fleet_growth = load.load_fleet_growth(model_excel_db)
    prev_year_fuel_group_avail = ref["prev_year_fuel_group_avail"]
    # Energy efficiency / speed-reduction tables (unfiltered; scenario filtering done in main)
    df_speed_ref = ref["df_speed_ref"]
    df_efficiency_ref = ref["df_efficiency_ref"]
    df_engine_efficiency_ref = ref["df_engine_efficiency_ref"]

    # --- processs CAPEX, retrofit cost ---
    df_capex_ref, df_retrofit_cost_ref, df_capex_reduction = load.load_capex_data(model_excel_db)
    df_capex_ref, df_retrofit_cost_ref = ship_econ.process_capex_retrofit_refs(user_params, df_capex_ref, df_retrofit_cost_ref,
                                                                               df_capex_reduction, end_year, planning_horizon_years)

    # --- FLEET ---
    assert 'random_seed' in user_params, f"'random_seed' not found in scenario row. Available: {list(user_params.keys())}"
    assert 'early_movers_fraction' in user_params, f"'early_movers_fraction' not found in scenario row. Available: {list(user_params.keys())}"
    df_vessels = load.load_rep_ships(user_params, model_excel_db, df_scrap_rates_ref, df_capex_ref)
    df_vessels = df_vessels.sample(n=num_players, replace=False, random_state=int(user_params['random_seed']))

    # Running the model with parameters from the scenario
    outputs = game_model.main(
        df_vessels, start_year, end_year, planning_horizon_years,
        su_market_flag, user_params,
        engines_design_mapping, engine_fuels_dict, df_drop_in_fuels_ref,
        df_energy_density_ref, df_imo_targets_data, df_fueleu_targets_data,
        ru_price_data, initial_su_price_data, fuel_price_data, df_growth_infra,
        df_capex_ref, df_retrofit_cost_ref,
        df_opex_conventional_ref, df_opex_nonconv_wages_ref, df_opex_nonconv_insurance_ref,
        df_opex_nonconv_ssf_ref, df_opex_nonconv_isf_ref, df_opex_nonconv_spares_ref,
        df_operational_data_ref, df_wtt_carbon_intensity_ref, df_ttw_carbon_intensity_ref,
        df_reward_ref, df_tc_rates_ref, df_pilot_fuel_input_ref,
        ENGINE_PEER_FACTOR, 
        critical_mass_engine, slope_engine, max_iterations,
        scenario_output_dir, round_nb,
        prev_year_fuel_group_avail=prev_year_fuel_group_avail,
        df_fleet_growth=df_fleet_growth,
        df_speed_ref=df_speed_ref,
        df_efficiency_ref=df_efficiency_ref,
        df_engine_efficiency_ref=df_engine_efficiency_ref,
        SHAPE_PEER_INFLUENCE=SHAPE_PEER_INFLUENCE,
    )

    # Collect results from the model output
    result = {
        "scenario_name": scenario_params['scenario_name'],
        "myopic_results": outputs.get("myopic_csv_path"),
        "nash_results": outputs.get("nash_csv_path"),
        "compare_results": outputs.get("compare_csv_path"),
        "df_dcf_opt": outputs.get("df_dcf_opt"),
        "df_results_opt": outputs.get("df_results_opt"),
        "df_rrc_fuel_opt": outputs.get("df_rrc_fuel_opt"),
        "df_nash_peer_tracking": outputs.get("df_nash_peer_tracking")
    }

    print('results saved to: ', scenario_output_dir)

    return result

# Run the scenarios in parallel
def run_scenarios_parallel(scenario_df, RUN_OUTPUT_DIR, model_excel_db, round_nb=None, max_workers=None, executor_type='thread'):
    results = []

    executor_cls = concurrent.futures.ProcessPoolExecutor if executor_type == 'process' else concurrent.futures.ThreadPoolExecutor

    with executor_cls(max_workers=max_workers) as executor:
        futures = []
        for _, row in scenario_df.iterrows():
            # Extract round_nb from the scenario row
            row_round_nb = row.get('round_nb', round_nb)
            assert row_round_nb is not None, f"round_nb not found in scenario row and no default provided. Row: {row.to_dict()}"
            
            future = executor.submit(
                run_scenario,
                row.to_dict(),
                RUN_OUTPUT_DIR,
                model_excel_db,
                row_round_nb
            )
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results

# Running one fuel price at a time

In [ ]:
# INPUT SCENARIOS FILE
scenarios_input_file = f"../output/fem_results/version_{version}/scenarios_config_fem_paper_v{version}.xlsx"

scenarios_to_run_tab = "scenarios_to_run"

# OUTPUT RESULTS SAVE LOCATION:
RUN_OUTPUT_DIR = f"../output/fem_results/version_{version}/"
RUN_OUTPUT_DIR = Path(RUN_OUTPUT_DIR)

# MODEL DATABASEc
model_excel_db = '../data/Model_Database.xlsx'

# PARALLEL SETTINGS
max_workers = os.cpu_count()
executor_type = 'thread'  # set to 'process' to use processes


In [ ]:
# Reload modules after edits
import su_market
import fuel
import fleet_adoption_game
import policy
import ship_econ
import load_data

importlib.reload(policy)
importlib.reload(fuel)
importlib.reload(ship_econ)
importlib.reload(su_market)
importlib.reload(load_data)
importlib.reload(fleet_adoption_game)

print("Modules reloaded: policy, fuel, ship_econ, su_market, load_data, fleet_adoption_game")

scenario_df = load_scenario_data(scenarios_input_file, scenarios_to_run_tab)

# Validate and display round_nb values from scenario data
assert 'round_nb' in scenario_df.columns, f"'round_nb' column not found in scenario data. Available columns: {scenario_df.columns.tolist()}"
print(f"Round numbers from scenario file: {scenario_df['round_nb'].unique().tolist()}")
print(f"Scenarios per round: {scenario_df.groupby('round_nb').size().to_dict()}")

results = run_scenarios_parallel(
    scenario_df,
    RUN_OUTPUT_DIR,
    model_excel_db,
    round_nb=None,  # round_nb will be extracted from each scenario row
    max_workers=max_workers,
    executor_type=executor_type,
)

print("Scenarios have been executed successfully.")


# Biofuel ramp-up scenarios
For each `year_biofuel_ramp_up` in range(2031, 2041), build a spliced fuel price table:
- Bio-fuel prices from **"Central scenario"** for years **before** the ramp-up year
- Bio-fuel prices from **"UMAS/UCL (2025)"** from the ramp-up year **onward**

All other scenario parameters are taken from the base row in the Excel config.

In [ ]:
# INPUT SCENARIOS FILE
scenarios_input_file = f"../output/fem_results/version_{version}/scenarios_config_fem_paper_v{version}.xlsx"

scenarios_to_run_tab = "scenarios_to_run"

# OUTPUT RESULTS SAVE LOCATION:
RUN_OUTPUT_DIR = f"../output/fem_results/version_{version}/"
RUN_OUTPUT_DIR = Path(RUN_OUTPUT_DIR)

# MODEL DATABASEc
model_excel_db = '../data/Model_Database.xlsx'

# PARALLEL SETTINGS
max_workers = os.cpu_count()
executor_type = 'thread'  # set to 'process' to use processes


In [ ]:
import importlib, copy
import su_market, fuel, fleet_adoption_game as game_model, policy, ship_econ, load_data as load

for mod in [policy, fuel, ship_econ, su_market, load, game_model]:
    importlib.reload(mod)

# --- Configuration ---
bio_scenario_before = "Central scenario"       # biofuel prices before ramp-up year
bio_scenario_after  = "UMAS/UCL (2025)"        # biofuel prices from ramp-up year onward
ramp_up_years = range(2031, 2041, 3)              # scenarios
ramp_up_years = [2034]

# Load base scenarios from Excel config (each row is a base scenario template)
scenario_df_base = load_scenario_data(scenarios_input_file, scenarios_to_run_tab)
all_base_params = [convert_columns_to_dict(row.to_dict()) for _, row in scenario_df_base.iterrows()]

# --- Load all reference data once (using first row's allowed_fuels as superset) ---
all_allowed_fuels = set()
for bp in all_base_params:
    all_allowed_fuels.update(bp['allowed_fuels'])
engine_fuels_dict, df_drop_in_fuels_ref = load.load_engine_fuels_dict(model_excel_db, list(all_allowed_fuels))
ref = load.load_reference_data(model_excel_db)
fuel_price_data_raw = ref["fuel_price_data"]
df_fleet_growth = load.load_fleet_growth(model_excel_db)

(df_opex_conventional_ref, df_opex_nonconv_wages_ref, df_opex_nonconv_insurance_ref,
 df_opex_nonconv_ssf_ref, df_opex_nonconv_isf_ref, df_opex_nonconv_spares_ref) = ref["df_opex_data"]

# --- CAPEX / retrofit (use longest horizon across base scenarios) ---
max_end_year = max(bp['end_year'] for bp in all_base_params)
max_planning = max(bp['planning_horizon_years'] for bp in all_base_params)
df_capex_ref, df_retrofit_cost_ref, df_capex_reduction = load.load_capex_data(model_excel_db)
df_capex_ref, df_retrofit_cost_ref = ship_econ.process_capex_retrofit_refs(
    all_base_params[0], df_capex_ref, df_retrofit_cost_ref,
    df_capex_reduction, max_end_year, max_planning,
)

# --- Prepare bio-fuel price lookup tables (indexed by year) ---
bio_cols = [c for c in fuel_price_data_raw.columns if "bio-" in c]
df_bio_central = (
    fuel_price_data_raw[fuel_price_data_raw["scenario"] == bio_scenario_before]
    .sort_values("year").set_index("year")[bio_cols]
)
df_bio_umas = (
    fuel_price_data_raw[fuel_price_data_raw["scenario"] == bio_scenario_after]
    .sort_values("year").set_index("year")[bio_cols]
)

# --- Run ramp-up sweep for each base scenario ---
results_rampup = []

for base_params in all_base_params:
    print(f"\n{'#'*60}\n  Base scenario: {base_params['scenario_number']}\n{'#'*60}")
    
    # Extract round_nb from base_params
    round_nb = base_params.get('round_nb')
    assert round_nb is not None, f"round_nb not found in base scenario {base_params['scenario_number']}"

    # Fleet (sampled per base scenario in case num_players differs)
    assert 'random_seed' in base_params, f"'random_seed' not found in base scenario {base_params['scenario_number']}"
    assert 'early_movers_fraction' in base_params, f"'early_movers_fraction' not found in base scenario {base_params['scenario_number']}"
    df_vessels = load.load_rep_ships(base_params, model_excel_db, ref["df_scrap_rates_ref"], df_capex_ref)
    df_vessels = df_vessels.sample(n=base_params['num_players'], replace=False, random_state=int(base_params['random_seed']))

    for year_ramp in ramp_up_years:
        scenario_label  = f"{base_params['scenario_name']}_biofuel_ramp_up_{year_ramp}"
        scenario_number = f"{base_params['scenario_number']}biofuelrampup{year_ramp}"
        print(f"\n{'='*60}\n  {scenario_number}  (bio prices switch at {year_ramp})\n{'='*60}")

        # 1. Build spliced fuel_price_data with a single synthetic scenario
        df_spliced = (
            fuel_price_data_raw[fuel_price_data_raw["scenario"] == base_params["fuel_price_scenario"]]
            .sort_values("year").copy()
        )
        # Splice bio columns: Central before ramp-up, UMAS from ramp-up onward
        df_bio_spliced = pd.concat([
            df_bio_central.loc[df_bio_central.index < year_ramp],
            df_bio_umas.loc[df_bio_umas.index >= year_ramp],
        ])
        df_spliced = df_spliced.set_index("year")
        df_spliced[bio_cols] = df_bio_spliced.reindex(df_spliced.index)[bio_cols]
        df_spliced = df_spliced.reset_index()
        df_spliced["scenario"] = scenario_label

        # 2. Override scenario identification (both point to the synthetic scenario
        #    so preprocess_reference_data's bio/non-bio split is effectively a no-op)
        user_params = copy.deepcopy(base_params)
        user_params["scenario_number"]        = scenario_number
        user_params["scenario_name"]          = scenario_label
        user_params["fuel_price_scenario"]    = scenario_label
        user_params["biofuel_price_scenario"] = scenario_label

        # 3. Zero out reward from (year_ramp + year_step) onward
        reward_cutoff_year = year_ramp + base_params['year_step']
        df_reward_scenario = ref["df_reward_ref"].copy()
        df_reward_scenario.loc[df_reward_scenario['year'] >= reward_cutoff_year, 'value'] = 0

        # 4. Output directory
        scenario_output_dir = Path(RUN_OUTPUT_DIR) / "zip_files" / f"scenario_{scenario_number}"
        scenario_output_dir.mkdir(parents=True, exist_ok=True)

        # 5. Run model (using round_nb from base_params)
        outputs = game_model.main(
            df_vessels.copy(),
            user_params['start_year'], user_params['end_year'], user_params['planning_horizon_years'],
            user_params['su_market_flag'], user_params,
            ref["engines_design_mapping"], engine_fuels_dict, df_drop_in_fuels_ref,
            ref["df_energy_density_ref"], ref["df_imo_targets_data"], ref["df_fueleu_targets_data"],
            ref["ru_price_data"], ref["initial_su_price_data"], df_spliced, ref["df_growth_infra"],
            df_capex_ref, df_retrofit_cost_ref,
            df_opex_conventional_ref, df_opex_nonconv_wages_ref, df_opex_nonconv_insurance_ref,
            df_opex_nonconv_ssf_ref, df_opex_nonconv_isf_ref, df_opex_nonconv_spares_ref,
            ref["df_operational_data_ref"], ref["df_wtt_carbon_intensity_ref"], ref["df_ttw_carbon_intensity_ref"],
            df_reward_scenario, ref["df_tc_rates_ref"], ref["df_pilot_fuel_input_ref"],
            user_params['ENGINE_PEER_FACTOR'], 
            user_params['critical_mass_engine'], user_params['slope_engine'], user_params['max_iterations'],
            scenario_output_dir, round_nb,
            prev_year_fuel_group_avail=copy.deepcopy(ref["prev_year_fuel_group_avail"]),
            df_fleet_growth=df_fleet_growth,
            df_speed_ref=ref["df_speed_ref"],
            df_efficiency_ref=ref["df_efficiency_ref"],
            df_engine_efficiency_ref=ref["df_engine_efficiency_ref"],
            SHAPE_PEER_INFLUENCE=user_params['SHAPE_PEER_INFLUENCE'],
        )

        results_rampup.append({
            "base_scenario": scenario_number,
            "scenario_number": scenario_number,
            "year_biofuel_ramp_up": year_ramp,
            **{k: outputs.get(k) for k in ["myopic_csv_path", "nash_csv_path", "compare_csv_path",
                                             "df_dcf_opt", "df_results_opt", "df_rrc_fuel_opt",
                                             "df_nash_peer_tracking"]},
        })
        print(f"  -> saved to {scenario_output_dir}")